# 🎬 Studio Video AI - Motor GPU em Segundo Plano
### Conecta a GPU NVIDIA T4 gratuita do Google Colab diretamente ao seu site da Vercel

**Como usar:**
1. Verifique se o ambiente está em **GPU T4** (Ambiente de execução -> Alterar tipo de ambiente -> GPU T4).
2. Clique no **Play ▶** abaixo.
3. Copie o link que aparecerá (ex: `https://xxxx.trycloudflare.com`).
4. Abra seu site na Vercel e cole no botão **⚡ Conectar Motor GPU**.
5. Pronto! Agora tudo é feito 100% no seu site da Vercel!

In [ ]:
# ==============================================================================
# 🚀 MOTOR GPU STUDIO VIDEO AI (SADTALKER 512PX + GFPGAN + CLOUD BRIDGE)
# ==============================================================================
import os, sys, subprocess, time, re

print('📦 1. Instalando bibliotecas neurais e dependências...')
!pip install -q basicsr-fixed
!pip install -q kornia facexlib yacs safetensors gradio flask flask-cors edge-tts
!pip install -q --no-build-isolation basicsr
!pip install -q gfpgan

%cd /content
if not os.path.exists('/content/SadTalker'):
    !git clone -b v1.1 https://github.com/camenduru/SadTalker

# 2. Correção definitiva de compatibilidade em Python
preprocess_file = '/content/SadTalker/src/face3d/util/preprocess.py'
if os.path.exists(preprocess_file):
    with open(preprocess_file, 'r') as f:
        content = f.read()
    content = content.replace('np.VisibleDeprecationWarning', 'UserWarning')
    with open(preprocess_file, 'w') as f:
        f.write(content)

# 3. Baixando os pesos da rede neural (SadTalker 512px + GFPGAN)
!apt -y install -qq aria2
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/SadTalker_V0.0.2_256.safetensors -d /content/SadTalker/checkpoints -o SadTalker_V0.0.2_256.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/SadTalker_V0.0.2_512.safetensors -d /content/SadTalker/checkpoints -o SadTalker_V0.0.2_512.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/mapping_00109-model.pth.tar -d /content/SadTalker/checkpoints -o mapping_00109-model.pth.tar
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/mapping_00229-model.pth.tar -d /content/SadTalker/checkpoints -o mapping_00229-model.pth.tar
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/GFPGANv1.4.pth -d /content/SadTalker/gfpgan/weights -o GFPGANv1.4.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/alignment_WFLW_4HG.pth -d /content/SadTalker/gfpgan/weights -o alignment_WFLW_4HG.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/detection_Resnet50_Final.pth -d /content/SadTalker/gfpgan/weights -o detection_Resnet50_Final.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/parsing_parsenet.pth -d /content/SadTalker/gfpgan/weights -o parsing_parsenet.pth

# 4. Baixa assets padrão e a API Bridge do repositório
!curl -s -L -o /content/personagem_maconaria.jpg https://raw.githubusercontent.com/grego921/studio-video-ai/main/personagem_maconaria.jpg
!curl -s -L -o /content/colab_api.py https://raw.githubusercontent.com/grego921/studio-video-ai/main/colab_api.py

# 5. Instala Cloudflared para criar o túnel HTTPS seguro
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# 6. Inicia o servidor API em background
import threading
def start_api():
    subprocess.run(['python', '/content/colab_api.py'])
threading.Thread(target=start_api, daemon=True).start()
time.sleep(3)

# 7. Abre o túnel Cloudflare e exibe o link para o site da Vercel
tunnel_proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    match = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

print('\n' + '='*70)
if public_url:
    print('🔥 MOTOR GPU CONECTADO E PRONTO PARA O SEU SITE DA VERCEL! 🔥')
    print('\n👉 COPIE ESTA URL DA GPU:')
    print(f'   {public_url}')
    print('\n👉 E COLE NO SEU SITE DA VERCEL NO BOTÃO "⚡ Conectar Motor GPU"!')
else:
    print('⚠️ Aguardando inicialização do túnel... Veja as linhas acima para a URL .trycloudflare.com')
print('='*70 + '\n')
tunnel_proc.wait()
